# Model Experiments

All validation uses **chronological** splits (`TimeSeriesSplit`).
Shuffled `KFold` lets the model train on the future and test on the past, which inflates scores;
experiment 5 shows the difference.

Target: next-day return (%). Baseline: always predict 0% change.

In [ ]:
import os, sys
# Find the project root (folder that contains src/) no matter where Jupyter was started
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, 'src')) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
%matplotlib inline
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, KFold, cross_val_score, learning_curve
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, AdaBoostRegressor, VotingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
from src.train_model import prepare_features_target, split_data
from src.evaluate import calculate_metrics
np.random.seed(42)
tscv = TimeSeriesSplit(n_splits=5)

## 1. Load data & split chronologically

In [ ]:
from src.pipeline import WARMUP_CALENDAR_DAYS
from src.data_loader import StockDataLoader
from src.preprocess import StockDataPreprocessor, get_feature_columns

SYMBOL, START, END = 'AAPL', '2019-01-01', pd.Timestamp.today().strftime('%Y-%m-%d')
fetch_start = (pd.Timestamp(START) - pd.Timedelta(days=WARMUP_CALENDAR_DAYS)).strftime('%Y-%m-%d')
raw, source = StockDataLoader(SYMBOL).fetch(SYMBOL, fetch_start, END, save=False)
print(f"Source: {source}  rows: {len(raw)}")

df = StockDataPreprocessor(raw, verbose=False).run_pipeline()
df = df[df['Date'] >= START].reset_index(drop=True)
feature_cols = get_feature_columns(df)
TARGET = 'target_return_1d'   # next-day return in %
print(f"Processed: {df.shape}, features: {len(feature_cols)}")

X, y, feature_cols = prepare_features_target(df, target_col=TARGET, verbose=False)
X_train, X_test, y_train, y_test = split_data(X, y)
baseline_rmse = np.sqrt(np.mean(y_test ** 2))
print(f"Train {X_train.shape}  Test {X_test.shape}  |  zero-change baseline RMSE: {baseline_rmse:.4f}%")

def report(name, y_pred):
    m = calculate_metrics(y_test, y_pred, is_return_target=True)
    return {'Model': name, 'Test_R2': m['R2'], 'Test_RMSE': m['RMSE'], 'Test_MAE': m['MAE'],
            'Dir_Acc_%': m['Directional_Accuracy'], 'Beats_Baseline': m['RMSE'] < baseline_rmse}

## 2. Experiment 1: XGBoost hyper-parameter search (time-series CV)

In [ ]:
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.3, 0.6, 1.0],
    'min_child_weight': [1, 5, 20],
}
random_search = RandomizedSearchCV(XGBRegressor(random_state=42, n_jobs=1, verbosity=0), param_dist,
                                   n_iter=15, cv=tscv, scoring='neg_mean_squared_error', random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)
print("Best params:", random_search.best_params_)
print(f"Best CV RMSE: {np.sqrt(-random_search.best_score_):.4f}%")
display(pd.DataFrame([report('Tuned XGBoost', random_search.best_estimator_.predict(X_test))]))

## 3. Experiment 2: compare model families

In [ ]:
models = {
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    'Lasso': make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=5000)),
    'ElasticNet': make_pipeline(StandardScaler(), ElasticNet(alpha=0.01, max_iter=5000)),
    'Decision Tree': DecisionTreeRegressor(max_depth=4, random_state=42),
    'KNN': make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=25)),
    'SVR': make_pipeline(StandardScaler(), SVR(C=1.0)),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=50, random_state=42),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    row = report(name, model.predict(X_test))
    row['Train_R2'] = r2_score(y_train, model.predict(X_train))
    rows.append(row)
results_df = pd.DataFrame(rows).sort_values('Test_RMSE')
display(results_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(results_df['Model'], results_df['Test_R2'], color='steelblue'); axes[0].set_title('Test R²')
axes[0].axvline(0, color='black', linewidth=1)
axes[1].barh(results_df['Model'], results_df['Test_RMSE'], color='coral'); axes[1].set_title('Test RMSE (%)')
axes[1].axvline(baseline_rmse, color='black', linestyle='--', label='zero-change baseline'); axes[1].legend()
for ax in axes: ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

## 4. Experiment 3: ensembles

In [ ]:
voting_model = VotingRegressor([
    ('ridge', make_pipeline(StandardScaler(), Ridge(alpha=10.0))),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42, n_jobs=1, verbosity=0)),
]).fit(X_train, y_train)

from src.ensemble_model import SuperEnsemble     # time-series-safe stacking (tree-only without TensorFlow)
stack = SuperEnsemble(use_lstm=False).train_ensemble(X_train, y_train)

ens_df = pd.DataFrame([report('Voting', voting_model.predict(X_test)),
                       report('Stacking (TS-safe)', stack.predict(X_test))])
display(ens_df.round(4))

## 5. Experiment 4: number of features

In [ ]:
subset_rows = []
for k in [5, 10, 20, 40, 80]:
    pipe = make_pipeline(SelectKBest(f_regression, k=k),
                         XGBRegressor(n_estimators=100, max_depth=3, random_state=42, n_jobs=1, verbosity=0))
    cv_rmse = np.sqrt(-cross_val_score(pipe, X_train, y_train, cv=tscv, scoring='neg_mean_squared_error')).mean()
    pipe.fit(X_train, y_train)
    row = report(f'top-{k}', pipe.predict(X_test)); row.update({'Num_Features': k, 'CV_RMSE': cv_rmse})
    subset_rows.append(row)
subset_df = pd.DataFrame(subset_rows)
display(subset_df[['Num_Features', 'CV_RMSE', 'Test_RMSE', 'Test_R2', 'Dir_Acc_%']].round(4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(subset_df['Num_Features'], subset_df['CV_RMSE'], marker='o', label='CV RMSE (train period)')
ax.plot(subset_df['Num_Features'], subset_df['Test_RMSE'], marker='s', label='Test RMSE')
ax.axhline(baseline_rmse, color='black', linestyle='--', label='baseline')
ax.set_xscale('log'); ax.set_xlabel('Number of features'); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## 6. Experiment 5: why shuffled KFold is wrong for time series
If shuffled KFold scores look much better than TimeSeriesSplit, that gap is leakage, not skill.

In [ ]:
model = XGBRegressor(n_estimators=100, max_depth=3, random_state=42, n_jobs=1, verbosity=0)
cv_rows = []
for name, cv in {'TimeSeriesSplit-5 (correct)': tscv,
                 'KFold-5 shuffled (leaky)': KFold(n_splits=5, shuffle=True, random_state=42)}.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1)
    cv_rows.append({'Strategy': name, 'Mean_R2': scores.mean(), 'Std_R2': scores.std(),
                    'Min_R2': scores.min(), 'Max_R2': scores.max()})
display(pd.DataFrame(cv_rows).round(4))

## 7. Experiment 6: learning curve (time-series CV)

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    XGBRegressor(n_estimators=100, max_depth=3, random_state=42, n_jobs=1, verbosity=0),
    X_train, y_train, train_sizes=np.linspace(0.2, 1.0, 6), cv=tscv,
    scoring='neg_root_mean_squared_error', n_jobs=-1)
plt.figure(figsize=(10, 6))
for scores, label, marker in [(-train_scores, 'Training RMSE', 'o'), (-val_scores, 'Validation RMSE', 's')]:
    plt.plot(train_sizes, scores.mean(axis=1), marker=marker, label=label)
    plt.fill_between(train_sizes, scores.mean(axis=1) - scores.std(axis=1),
                     scores.mean(axis=1) + scores.std(axis=1), alpha=0.2)
plt.axhline(baseline_rmse, color='black', linestyle='--', label='test baseline')
plt.xlabel('Training set size'); plt.ylabel('RMSE (%)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Learning curve - XGBoost'); plt.show()

## 8. Summary

In [ ]:
all_results = pd.concat([results_df, ens_df], ignore_index=True).sort_values('Test_RMSE')
best = all_results.iloc[0]
summary = {
    'Best XGBoost params': random_search.best_params_,
    'Best model (test RMSE)': best['Model'],
    'Best test RMSE': round(best['Test_RMSE'], 4),
    'Zero-change baseline RMSE': round(baseline_rmse, 4),
    'Beats baseline': bool(best['Beats_Baseline']),
    'Best direction accuracy': round(all_results['Dir_Acc_%'].max(), 1),
    'Best feature count (CV)': int(subset_df.loc[subset_df['CV_RMSE'].idxmin(), 'Num_Features']),
}
for k, v in summary.items():
    print(f"{k:28s}: {v}")

## Conclusions & next steps
- Treat a model as useful only if it beats the zero-change baseline **out of sample**, consistently across symbols.
- Next: walk-forward backtest with transaction costs, other horizons (`target_return_5d`), more symbols.